# 00 — Data catalog (single entry point)

One grep-able index of every data artifact in the repo. For each file: where it lives (relative to repo root), who produced it, who reads it, and what its columns mean.

**How to use it.**
- Shell:  `grep -i "hardened_claim" notebooks/00_data_catalog.ipynb`
- Browser: use the TOC below and follow the anchors.
- Every path is RELATIVE to repo root, so grep results are portable.

We auto-scan the filesystem where we can and hand-annotate producer/consumer info where meaning matters. Nothing here modifies data or notebook files.

_(Notebook auto-generated. Self-contained: no figures — the value is the text tables.)_


> **Reader guide — where this notebook sits in the study.**
>
> **Part of:** *Orientation section* — no experiment; purpose is to give reviewers a single
> grep-able index of every data artefact and every notebook the study produced.
>
> **Question this NB answers:** *where does every file in `data/`, `figures/`, and `tables/` come from,
> which notebook wrote it, and which notebooks consume it?*
>
> **Method:** auto-scan of the filesystem + hand-annotated producer/consumer table per file.
>
> **Reproducibility contract:** this notebook only READS filenames + sizes from disk; no science
> is computed here. Every file it lists is either shipped in-repo or documented in
> `data/external/README.md` as a fetch step. Re-execute after any pull to refresh the index.
>
> **Environment:** requires `pip install -e .` from the repo root (or `pixi run` from `env/`)
> so that `import gbsabench` and `import discovery9` resolve.


## Table of contents

1. [Reading order (start here)](#sec-reading-order)
2. [Raw data — per-complex analyses and upstream raw CSVs](#sec-raw)
3. [Derived data — features, ligand-chem, baselines, sweeps](#sec-derived)
4. [Tables — publication-facing rollups](#sec-tables)
5. [Figures — downstream + upstream + SI](#sec-figures)
6. [External data — gbsa-study package](#sec-external)
7. [Notebook index — reverse map: NB → consumes / produces](#sec-nbindex)
8. [Quick-reference cheat sheet](#sec-cheat)
9. [Known inconsistencies (schema drift, doc drift)](#sec-drift)
10. [Orphans + unresolved items](#sec-orphans)
11. [Off-repo storage — STORE tree, workspaces](#sec-offrepo)


In [ ]:
# --- notebook preamble (matches NBs 11-19) ---
NB_STEM = "00_data_catalog"

import sys, os, json, glob
from pathlib import Path

# Make the in-repo src package importable without an install
# Find repo root robustly (walks up from CWD to the folder that contains pyproject.toml).
_repo_root = Path.cwd()
while _repo_root != _repo_root.parent and not (_repo_root / 'pyproject.toml').is_file():
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from discovery9.style import apply_style, NAVY, GOLD, GREY, GREY_DASH as GREYD, CREAM, WHITE
from discovery9.paths import ROOT, RAW, DERIVED, EXTERNAL, FIGURES, TABLES, GBSA_STUDY
apply_style()

# --- fig-capture hook (kept identical to downstream NBs; this NB may render 0 or 1 fig) ---
_SAVED_FIGS = globals().setdefault('_SAVED_FIGS', [])
_orig_figure = plt.figure
_orig_subplots = plt.subplots
def _figure_capture(*a, **kw):
    fig = _orig_figure(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig
def _subplots_capture(*a, **kw):
    fig, ax = _orig_subplots(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig, ax
plt.figure = _figure_capture
plt.subplots = _subplots_capture

print('ROOT:', ROOT)
print('all paths printed below are RELATIVE to ROOT.')

In [ ]:
# --- helpers used across sections ---

def rel(p):
    """Path -> str relative to ROOT (portable, grep-friendly)."""
    try:
        return str(Path(p).resolve().relative_to(ROOT.resolve()))
    except Exception:
        return str(p)

def size_mb(p):
    try:
        return os.path.getsize(p) / (1024 * 1024)
    except Exception:
        return float('nan')

def size_kb(p):
    try:
        return os.path.getsize(p) / 1024
    except Exception:
        return float('nan')

def n_rows_csv(p, sep=',', comment=None):
    """Row count for a CSV/TSV. Returns int or None. Uses comment='#' when needed."""
    try:
        return int(len(pd.read_csv(p, sep=sep, comment=comment)))
    except Exception:
        return None

def cols_csv(p, n=6, sep=',', comment=None):
    try:
        return list(pd.read_csv(p, sep=sep, nrows=1, comment=comment).columns)[:n]
    except Exception:
        return []

def shape_parquet(p):
    try:
        df = pd.read_parquet(p)
        return df.shape, list(df.columns)
    except Exception:
        return (None, None), []

def print_row(cols, widths):
    """Print one dense table row for grep-ability."""
    parts = []
    for c, w in zip(cols, widths):
        s = str(c) if c is not None else ''
        if len(s) > w:
            s = s[: max(0, w - 1)] + '…'
        parts.append(s.ljust(w))
    print(' | '.join(parts).rstrip())

def hdr(cols, widths):
    print_row(cols, widths)
    print('-+-'.join('-' * w for w in widths))

print('helpers ready.')

<a id="sec-reading-order"></a>
## 1. Reading order (start here)

If you're new, read in this order:

1. **`README.md`** — what the study is, what it found, repo layout.
2. **`STUDY_DESIGN.md`** — cohort (NewBench-27), locked protocol, cross-validation, scope.
3. **`notebooks/01_scope_and_map.ipynb`** — the 9 discovery + 18 validation split, plus a question-to-notebook map.
4. **`notebooks/00_data_catalog.ipynb`** (this one) — where every file lives and who touches it.
5. Any downstream NB (`notebooks/03..30`, the analytical slots after 00/01/02) for the question you care about.
6. For the upstream GBSA-physics / GROMACS-config context, those NBs now live in-tree at slot 02 and slots 04-13.

Every derived table has a regeneration script in `reproduce/`. Every publication rollup lives in `tables/`.

`reviews/` holds the iteration logs (iter_01 … iter_07). If you're joining mid-review, start with `reviews/STUDY_BRIEF.md`.


<a id="sec-raw"></a>
## 2. Raw data

Two sources:

- **`data/raw/complex_analyses/<target>/<complex_id>/`** — one directory per (target, ligand) with per-frame MD outputs. 9 discovery targets × 30 ligands = 270 complexes. Each contains the same fixed set of files.
- **`data/external/gbsa-study/data/raw/`** — the upstream GBSA-study raw campaign CSVs (per-complex GBSA ΔG, docking scores, activity labels, MD wallclock). READ-ONLY from downstream.

`complex_analyses/` is read by `notebooks/14_per_complex_deep_dive.ipynb` (via `load_per_complex_analysis`) and aggregated by `reproduce/aggregate_results.py` into `data/derived/features.parquet`.


In [ ]:
# 2a. per-complex analysis tree ----------------------------------------------
print('### 2a. `data/raw/complex_analyses/<target>/<complex_id>/`  --  the 270 per-complex analysis dirs\n')
targets = sorted([d.name for d in RAW.joinpath('complex_analyses').iterdir() if d.is_dir()])
print(f'discovery targets  ({len(targets)}) : {targets}')
n_complexes = 0
for t in targets:
    n_complexes += sum(1 for _ in (RAW / 'complex_analyses' / t).iterdir() if _.is_dir())
print(f'total complex dirs : {n_complexes}   (expected: 9 targets x 30 ligands = 270)')

# Pick a sample and describe the fixed file set that every complex contains.
sample = next((RAW / 'complex_analyses' / targets[0]).iterdir())
sample_files = sorted([p.name for p in sample.iterdir()])
print(f'\nsample complex     : data/raw/complex_analyses/{targets[0]}/{sample.name}/')
print('files (every complex has the same set):')
for f in sample_files:
    print(f'    {f}')
print('\nwhat each file is:')
descrip = [
    ('summary.json',            'per-complex scalar summary: n_frames, drift, PBC flag, active-site atoms, etc.'),
    ('timeseries.parquet',      'per-frame kinematic descriptors (RMSD, RMSF, contacts, SASA, ligand COM disp, ...)'),
    ('hbond_timeseries.parquet','per-frame H-bond counts (ligand-protein, ligand-water)'),
    ('ifp_timeseries.parquet',  'per-frame interaction fingerprint (bit vector over active-site residues)'),
    ('ligand_rmsf.parquet',     'per-atom ligand RMSF (heavy atoms)'),
    ('protein_ca_rmsf.parquet', 'per-residue protein-Ca RMSF'),
    ('contacts_persistence.tsv','per-residue contact persistence fraction (ligand-protein)'),
    ('hbonds_persistence.tsv',  'per-donor-acceptor H-bond persistence fraction'),
]
for name, desc in descrip:
    print(f'    {name:28s}  {desc}')
print('\nProducer : reproduce/analyze_complex.py  (run once per complex on the raw MD trajectory)')
print('Aggregator: reproduce/aggregate_results.py  -->  data/derived/features.parquet + features.tsv')
print('Consumers: notebooks/14_per_complex_deep_dive.ipynb (loads timeseries / rmsf / contacts directly)')
print('           notebooks/19_active_site_fingerprint.ipynb (loads contacts_persistence.tsv)')
print('           reproduce/pbc_qc.py (walks all 270 timeseries.parquet)')

In [ ]:
# 2b. upstream raw CSVs ------------------------------------------------------
print('### 2b. `data/external/gbsa-study/data/raw/`  --  upstream campaign raw CSVs\n')

URAW = GBSA_STUDY / 'data' / 'raw'

# One row per file. Producer = the upstream project. Consumer info hand-curated.
entries = [
    dict(
        file='data/external/gbsa-study/data/raw/newbench_targets.csv',
        flag='STUDY-SCOPE MASTER LIST',
        pk='pdb',
        what='NewBench-27 target list: 9 discovery + 18 validation. Every scope decision references this file.',
        consumers='notebooks/01_scope_and_map.ipynb',
    ),
    dict(
        file='data/external/gbsa-study/data/raw/metadata.csv',
        flag='ACTIVES/DECOYS SOURCE OF TRUTH (fallback when MANIFEST.tsv is empty)',
        pk='(complex_id, target)',
        what='per-complex ligand identity + activity label + docking score. is_active in {True, False}.',
        consumers='discovery9.io.load_metadata; NB 10, 12, 15, 16, 17, 18; reproduce/{canonical_baselines,deep_research,gbsa_param_vs_md_v2,generate_summary_tables}.py; reproduce/upstream/{bedroc_doe,temporal_convergence}.py',
    ),
    dict(
        file='data/external/gbsa-study/data/raw/gbsa_dG_raw.csv',
        flag='PRIMARY GBSA MEASUREMENT',
        pk='(complex_id, target, combo)',
        what='per-complex mean MM-GBSA dG (kcal/mol) at each of the 48 GBSA parameter combos. Renamed to gbsa_dG on load.',
        consumers='discovery9.io.load_gbsa / load_gbsa_all; NB 10, 12, 15, 16, 17; reproduce/{canonical_baselines,deep_research,gbsa_param_vs_md_v2}.py; reproduce/upstream/{bedroc_doe,temporal_convergence}.py',
    ),
    dict(
        file='data/external/gbsa-study/data/raw/md_productions_raw.csv',
        flag='',
        pk='(complex_id, target)',
        what='per-complex MD production wallclock + ns/day + success flag from the primary discovery campaign.',
        consumers='notebooks/04_coverage.ipynb; notebooks/11_timestep_throughput.ipynb (context)',
    ),
    dict(
        file='data/external/gbsa-study/data/raw/md_variants_manifest_raw.csv',
        flag='',
        pk='(config, target, complex_id)',
        what='study-2 L27 config manifest: which (config, complex) combinations were attempted.',
        consumers='notebooks/09_study2_gromacs_screening.ipynb',
    ),
    dict(
        file='data/external/gbsa-study/data/raw/md_variants_prod_perf_raw.csv',
        flag='',
        pk='(config, target, complex_id)',
        what='study-2 L27 production performance: wallclock, ns/day, success/fail per (config, complex).',
        consumers='notebooks/09_study2_gromacs_screening.ipynb',
    ),
    dict(
        file='data/external/gbsa-study/data/raw/md_variants_gbsa_scores_raw.csv',
        flag='',
        pk='(config, target, complex_id)',
        what='study-2 L27 GBSA rescores per (config, complex).',
        consumers='notebooks/09_study2_gromacs_screening.ipynb',
    ),
    dict(
        file='data/external/gbsa-study/data/raw/md_variants_gbsa_scores_raw.csv.pre-rerun-backup',
        flag='BACKUP -- do not consume',
        pk='n/a',
        what='pre-rerun backup of md_variants_gbsa_scores_raw.csv. Kept for provenance, not used by any NB.',
        consumers='(none)',
    ),
    dict(
        file='data/external/gbsa-study/data/raw/md_speed_smoke_L40S.csv',
        flag='',
        pk='n/a',
        what='single-GPU L40S smoke test for timestep/MTS parameter sweep.',
        consumers='notebooks/11_timestep_throughput.ipynb; notebooks/10_timestep_stability.ipynb',
    ),
]

widths = [72, 12, 12, 34, 60]
hdr(['file (relative)', 'size MB', 'n_rows', 'flag', 'first columns'], widths)
for e in entries:
    p = ROOT / e['file']
    print_row([e['file'], f"{size_mb(p):.2f}", n_rows_csv(p) or '', e['flag'], ', '.join(cols_csv(p))], widths)

print('\n--- primary-key + purpose ---')
for e in entries:
    print(f"  {e['file']}")
    print(f"      pk       : {e['pk']}")
    print(f"      purpose  : {e['what']}")
    print(f"      consumers: {e['consumers']}")


<a id="sec-derived"></a>
## 3. Derived data (`data/derived/`)

Aggregated per-complex or per-target tables. Every file below has a producer script in `reproduce/` and a list of consumer notebooks (grepped from `notebooks/`).

Paths are always relative to repo root.


In [ ]:
# 3. downstream derived --------------------------------------------------------
print('### 3. downstream `data/derived/*`\n')

# Hand-curated producer + consumer info. Consumer lists are grepped: any NB that
# either directly reads the file OR imports it through discovery9.io.load_features()
# is counted as a consumer of features.parquet + ligand_chem.parquet.
derived_entries = [
    dict(
        file='data/derived/features.parquet',
        producer='reproduce/aggregate_results.py',
        consumers='discovery9.io.load_features; NB 03,02,03,04,05,06,07,08,09,10,11,12,13,14,15,16,17,18 (via load_features); reproduce/{canonical_baselines,deep_research,gbsa_param_vs_md,gbsa_param_vs_md_v2,generate_summary_tables}.py',
        key='(target, complex_id)',
        what='wide per-complex MD-feature table: 270 rows x ~68 columns (kinematic descriptors, contacts, SASA, active-site metrics).',
    ),
    dict(
        file='data/derived/features.tsv',
        producer='reproduce/aggregate_results.py (same aggregator writes both .parquet and .tsv)',
        consumers='reproduce/canonical_baselines.py (reads the TSV as a legibility check)',
        key='(target, complex_id)',
        what='human-readable mirror of features.parquet. Kept in sync by aggregate_results.py.',
    ),
    dict(
        file='data/derived/ligand_chem.parquet',
        producer='reproduce/ligand_chem_from_topology.py',
        consumers='discovery9.io.load_features (auto-merges when present); NB 27,17 (explicit); reproduce/{canonical_baselines,gbsa_param_vs_md_v2,generate_summary_tables}.py',
        key='(target, complex_id)',
        what='ligand-only chemical descriptors (MW, LogP, TPSA, HBD, HBA, rot_bonds, aromatic_rings, ...). 270 x 15.',
    ),
    dict(
        file='data/derived/pbc_qc.csv',
        producer='reproduce/pbc_qc.py',
        consumers='reviews/iter_04 + iter_05 QC pass (referenced in reviewer briefs); not currently consumed by any NB',
        key='(target, complex_id)',
        what='per-trajectory PBC / drift QC flags: max COM jump, max drift, is_pbc_artifact, is_probably_unbound. 270 rows.',
    ),
    dict(
        file='data/derived/canonical_baselines.csv',
        producer='reproduce/canonical_baselines.py',
        consumers='NB 22,11,12,13,16,17,18,19 (single-source-of-truth check for baseline BEDROC values)',
        key='baseline_name',
        what='SINGLE SOURCE OF TRUTH for canonical baseline panel BEDROC alpha=20 values (gbsa_locked_9T_imputed, gbsa_locked_8T_excluded, docking_9T, top-feature naive + MW-residualised). CSV header uses `#`-comment lines; load with pd.read_csv(comment="#").',
    ),
    dict(
        file='data/derived/deep_research_summary.csv',
        producer='reproduce/deep_research.py',
        consumers='reproduce/generate_summary_tables.py',
        key='approach',
        what='deep-research P1/P2/P3 panel-BEDROC summary (one row per approach).',
    ),
    dict(
        file='data/derived/deep_research_wide.csv',
        producer='reproduce/deep_research_wide.py',
        consumers='NB 23,12 (protocol P1/P2 wider ML sweep); NB 30 (family stratification consumes it); reproduce/generate_summary_tables.py',
        key='(protocol, model)',
        what='wider LOTO ML sweep: per (protocol P1|P2, model) panel BEDROC + 95% bootstrap CI + permutation p + per-target string.',
    ),
    dict(
        file='data/derived/deep_research_p1_per_target.csv',
        producer='reproduce/deep_research.py',
        consumers='(rendered only -- referenced in NB 25 verdict)',
        key='target',
        what='per-target P1 BEDROC for oracle, locked_sp, mean, RidgeCV, RandomForest, HGBT.',
    ),
    dict(
        file='data/derived/deep_research_p2_per_target.csv',
        producer='reproduce/deep_research.py',
        consumers='(rendered only -- referenced in NB 25 verdict)',
        key='target',
        what='per-target P2 BEDROC for constant, logistic, RF, HGBT, MD-composite, docking, gbsa_locked.',
    ),
    dict(
        file='data/derived/deep_research_p3_per_target.csv',
        producer='reproduce/deep_research.py',
        consumers='(rendered only -- referenced in NB 25 verdict)',
        key='target',
        what='per-target P3 picks_combo, picks_bedroc, ensemble_bedroc.',
    ),
    dict(
        file='data/derived/hardened_claim_b.csv',
        producer='reproduce/hardened_claim_b.py',
        consumers='NB 28 (single-feature BEDROC) and NB 01 (scope map references it); reproduce/generate_summary_tables.py',
        key='(feature, condition, sign)',
        what='hardened Claim-B evidence: per-feature BEDROC vs GBSA-on-subset, under multiple label conditions.',
    ),
    dict(
        file='data/derived/hardened_claim_b_summary.md',
        producer='reproduce/hardened_claim_b.py (also emits the .md alongside the CSV)',
        consumers='(human read only)',
        key='n/a',
        what='human-readable summary of the hardened-Claim-B result (verdict paragraph + top rows).',
    ),
    dict(
        file='data/derived/gbsa_param_vs_md_correlations.csv',
        producer='reproduce/gbsa_param_vs_md.py',
        consumers='NB 26 (gbsa param vs MD); NB 01 (map references it)',
        key='(feature, axis, pair)',
        what='v1 Spearman(MD-feature, dBEDROC per GBSA parameter axis) with permutation p.',
    ),
    dict(
        file='data/derived/gbsa_param_vs_md_v2_correlations.csv',
        producer='reproduce/gbsa_param_vs_md_v2.py',
        consumers='NB 26 (v2 tightened correlations)',
        key='(feature, axis, is_lig_chem)',
        what='v2 Spearman with BH-corrected q; feature x GBSA-axis long-form.',
    ),
    dict(
        file='data/derived/gbsa_param_vs_md_v2_q_matrix.csv',
        producer='reproduce/gbsa_param_vs_md_v2.py',
        consumers='NB 26',
        key='feature',
        what='v2 BH-corrected q-value matrix: feature (rows) x GBSA axis (cols).',
    ),
    dict(
        file='data/derived/gbsa_param_vs_md_v2_rho_matrix.csv',
        producer='reproduce/gbsa_param_vs_md_v2.py',
        consumers='NB 26',
        key='feature',
        what='v2 Spearman-rho matrix: feature (rows) x GBSA axis (cols).',
    ),
    dict(
        file='data/derived/gbsa_prediction_r_by_target.csv',
        producer='reproduce/gbsa_param_vs_md_v2.py (secondary output)',
        consumers='NB 27 (ligand_chem GBSA surrogate)',
        key='target',
        what='per-target Pearson r for MD-only, lig-only and MD+lig regressors predicting GBSA dG.',
    ),
    dict(
        file='data/derived/md_surrogate_bedroc.csv',
        producer='UNKNOWN -- investigate (no producer script found in reproduce/; may be a stale artifact from NB 27 hand-run)',
        consumers='(none in current NB tree)',
        key='target',
        what='per-target BEDROC for docking, GBSA_locked, MD-surrogate, MD+lig-surrogate on n_labeled ligands.',
    ),
    dict(
        file='data/derived/single_feature_bedroc.csv',
        producer='UNKNOWN -- investigate (no producer script found in reproduce/; likely written by NB 28 in an earlier run)',
        consumers='NB 28 (single-feature BEDROC); NB 30 (family stratification uses it); NB 01',
        key='feature',
        what='per-feature panel BEDROC (global vs oracle-signed direction) + count of targets beating GBSA.',
    ),
    dict(
        file='data/derived/deep_research_wide_summary.md',
        producer='reproduce/deep_research_wide.py (secondary .md output alongside CSV)',
        consumers='(human read only)',
        key='n/a',
        what='human-readable summary of the wide LOTO ML sweep.',
    ),
]

widths = [56, 12, 10, 12, 40]
hdr(['file (relative)', 'size MB', 'n_rows', 'key', 'first columns'], widths)
for e in derived_entries:
    p = ROOT / e['file']
    if e['file'].endswith('.parquet'):
        (shape, cols) = shape_parquet(p)
        rows = shape[0] if shape else None
        col_str = ', '.join(cols[:5]) if cols else ''
    elif e['file'].endswith('.md'):
        rows = None
        col_str = '(markdown)'
    elif e['file'].endswith('.tsv'):
        rows = n_rows_csv(p, sep='\t')
        col_str = ', '.join(cols_csv(p, sep='\t'))
    elif 'canonical_baselines' in e['file']:
        rows = n_rows_csv(p, comment='#')
        col_str = ', '.join(cols_csv(p, comment='#'))
    else:
        rows = n_rows_csv(p)
        col_str = ', '.join(cols_csv(p))
    print_row([e['file'], f"{size_mb(p):.3f}", rows or '', e['key'], col_str], widths)

print('\n--- producer / consumer / purpose ---')
for e in derived_entries:
    print(f"  {e['file']}")
    print(f"      producer : {e['producer']}")
    print(f"      consumers: {e['consumers']}")
    print(f"      what     : {e['what']}")

<a id="sec-tables"></a>
## 4. Tables (`tables/`)

Publication-facing rollups. Everything here is regenerated by `reproduce/generate_summary_tables.py`. By convention each file has a `#`-comment header with the regenerate command.


In [ ]:
print('### 4. `tables/*.csv`\n')

table_entries = [
    dict(
        file='tables/panel_bedroc_summary.csv',
        producer='reproduce/generate_summary_tables.py',
        consumers='NB 01 (scope map cross-reference); publication figure sources',
        key='(score, condition)',
        what='downstream headline panel BEDROC per ranker, across labelling / alpha conditions. After iter 5-6 baseline reconcile, this file contains canonical rows AND `_8T_legacy` rows for the older 8T-excluded runs -- both are labelled and coexist.',
    ),
    dict(
        file='tables/rank_fusion_sweep.csv',
        producer='reproduce/generate_summary_tables.py',
        consumers='NB 29 (rank fusion deployable) verdict cross-check; NB 01 (scope map)',
        key='(K, models)',
        what='LOTO Borda rank-fusion over top-K single features. K vs panel BEDROC + 95% CI.',
    ),
]

widths = [46, 12, 10, 20, 60]
hdr(['file (relative)', 'size MB', 'n_rows', 'key', 'first columns'], widths)
for e in table_entries:
    p = ROOT / e['file']
    rows = n_rows_csv(p, comment='#')
    print_row([e['file'], f"{size_mb(p):.3f}", rows or '', e['key'], ', '.join(cols_csv(p, comment='#'))], widths)

print('\n--- producer / consumer / purpose ---')
for e in table_entries:
    print(f"  {e['file']}")
    print(f"      producer : {e['producer']}")
    print(f"      consumers: {e['consumers']}")
    print(f"      what     : {e['what']}")

print('\nNote: `data/derived/canonical_baselines.csv` is the SSoT for the baseline BEDROC values; it lives in derived/, not tables/. See section 9.')

<a id="sec-figures"></a>
## 5. Figures

Three subtrees:

- `figures/` — rendered by NBs 00..30 in the flat tree. One PNG per (NB, figN); the filename encodes the producer NB.
- `figures/upstream/` — legacy folder from before the flatten (subdirs `doe/`, `study2/`, `temporal/`). After the flatten pass these are still referenced by upstream-origin NBs at slot 02 and slots 04-13. Captions live in `figures/upstream/CAPTIONS.md`.
- `figures/SI/` — per-target Supplementary Information panels, grouped by target and by analysis. Also has a bundled PDF (`figures/SI/pdf/discovery9_SI_all.pdf`) built by `reproduce/bundle_SI_pdf.py`.


In [ ]:
print('### 5a. downstream `figures/*.png`  (one per NB fig; NB prefix maps to producer)\n')

# Load upstream captions if present.
cap_path = FIGURES / 'upstream' / 'CAPTIONS.md'
captions = {}
if cap_path.exists():
    for line in cap_path.read_text().splitlines():
        # lines look like '- `file.png` -- **NB_name** -- caption text' or '- **NB_name** -- suppressed title: ...'
        if '`' in line and '.png' in line:
            try:
                fname = line.split('`')[1]
                cap = line.split('--', 2)[-1].strip() if '--' in line else ''
                captions[Path(fname).name] = cap
            except Exception:
                pass

def caption_for(fname):
    return captions.get(fname, '')

downstream_pngs = sorted([p for p in FIGURES.glob('*.png')])
widths = [46, 10, 60]
hdr(['file (relative)', 'KB', 'source NB (from filename prefix)'], widths)
for p in downstream_pngs:
    stem = p.name.rsplit('_fig', 1)[0]
    print_row([rel(p), f'{size_kb(p):.0f}', f'notebooks/{stem}.ipynb'], widths)
print(f'\ntotal downstream PNGs: {len(downstream_pngs)}')

In [ ]:
print('### 5b. `figures/upstream/`  (upstream NBs 02..10 -- captions in figures/upstream/CAPTIONS.md)\n')

up_pngs = sorted([p for p in FIGURES.joinpath('upstream').rglob('*.png')])
widths = [66, 10, 40, 60]
hdr(['file (relative)', 'KB', 'source NB', 'caption (first 60 char)'], widths)
for p in up_pngs:
    stem = p.name.rsplit('_fig', 1)[0] if '_fig' in p.name else p.name.rsplit('.', 1)[0]
    # NB-prefix = the two-digit prefix.
    nb_prefix = p.name.split('_', 1)[0]
    # find matching upstream nb
    nb_match = [x for x in os.listdir(ROOT / 'notebooks' / 'upstream') if x.startswith(nb_prefix + '_')]
    nb = f"notebooks/upstream/{nb_match[0]}" if nb_match else f'(nb {nb_prefix} not found)'
    cap = caption_for(p.name)
    print_row([rel(p), f'{size_kb(p):.0f}', nb, cap], widths)
print(f'\ntotal upstream PNGs: {len(up_pngs)}')
print(f'PDF companions coexist: {len(list(FIGURES.joinpath("upstream").rglob("*.pdf")))} .pdf files.')

In [ ]:
print('### 5c. `figures/SI/`  (per-target Supplementary Information)\n')

si_root = FIGURES / 'SI'
top_files = sorted([p for p in si_root.glob('*.png')])
widths = [44, 10, 60]
hdr(['file (relative)', 'KB', 'note'], widths)
for p in top_files:
    print_row([rel(p), f'{size_kb(p):.0f}', 'top-of-SI overview'], widths)

# Per-target directories
per_target_dirs = sorted([d for d in si_root.iterdir() if d.is_dir() and d.name != 'pdf'])
print(f'\nper-target SI dirs ({len(per_target_dirs)}): {[d.name for d in per_target_dirs]}')
if per_target_dirs:
    sample_files = sorted([p.name for p in per_target_dirs[0].glob('*.png')])
    print(f'files per target (using {per_target_dirs[0].name} as sample):')
    for f in sample_files:
        print(f'    {f}')

# PDFs
pdfs = sorted(si_root.joinpath('pdf').glob('*.pdf'))
print(f'\nSI PDFs ({len(pdfs)}):')
for p in pdfs:
    print(f'    {rel(p)}   ({size_kb(p):.0f} KB)')
print('\nProducer: reproduce/bundle_SI_pdf.py + reproduce/make_SI_per_target.py')

In [ ]:
# Optional single figure: which NB reads which derived/tables/raw file.
# Grepped from the notebooks/ tree in-memory. NAVY = reads; GREY = does not read.
print('rendering the "which NB reads which file" heatmap...')

# Files to include on the x-axis (a curated short list; skip parquets internally loaded by discovery9.io because
# every downstream NB imports them via load_features).
files_axis = [
    'data/raw/complex_analyses',                                # timeseries etc.
    'data/derived/features.parquet',
    'data/derived/ligand_chem.parquet',
    'data/derived/canonical_baselines.csv',
    'data/derived/deep_research_wide.csv',
    'data/derived/hardened_claim_b.csv',
    'data/derived/single_feature_bedroc.csv',
    'data/derived/gbsa_param_vs_md_v2_correlations.csv',
    'data/external/gbsa-study/data/raw/metadata.csv',
    'data/external/gbsa-study/data/raw/gbsa_dG_raw.csv',
    'data/external/gbsa-study/data/raw/newbench_targets.csv',
    'data/external/gbsa-study/data/derived/study2/bedroc20_partial.csv',
    'data/external/gbsa-study/data/derived/temporal/bedroc_all_combos_per_target.csv',
]

# All downstream NBs (00-30) in a stable order.
# rglob catches the experiment-subdir layout (00_orientation/, A1_.../, A2_.../, ...)
downstream_nbs = sorted([p for p in (ROOT / 'notebooks').rglob('*.ipynb')
                         if '.ipynb_checkpoints' not in p.parts])

M = np.zeros((len(downstream_nbs), len(files_axis)), dtype=int)
for i, nb in enumerate(downstream_nbs):
    text = nb.read_text()
    for j, f in enumerate(files_axis):
        needle = f.split('/')[-1] if f.endswith('.csv') or f.endswith('.parquet') else f.split('/')[-1]
        if needle in text:
            M[i, j] = 1
        # Special: any NB that calls load_features / load_metadata / load_gbsa is implicitly a consumer of the
        # underlying raw+derived files. Encode this so the heatmap is not misleadingly sparse.
        if f == 'data/derived/features.parquet' and 'load_features(' in text:
            M[i, j] = 1
        if f == 'data/external/gbsa-study/data/raw/metadata.csv' and 'load_metadata(' in text:
            M[i, j] = 1
        if f == 'data/external/gbsa-study/data/raw/gbsa_dG_raw.csv' and ('load_gbsa(' in text or 'load_gbsa_all(' in text):
            M[i, j] = 1
        if f == 'data/raw/complex_analyses' and 'load_per_complex_analysis(' in text:
            M[i, j] = 1


# --- persist the underlying plot matrix so reviewers can rebuild the heatmap without re-executing ---
_read_df = pd.DataFrame(
    M,
    index=[p.stem for p in downstream_nbs],
    columns=[f.split('/')[-1] for f in files_axis],
)
DERIVED.mkdir(parents=True, exist_ok=True)
_read_df.to_csv(DERIVED / f"{NB_STEM}_read_matrix.csv")
print(f"wrote data/derived/{NB_STEM}_read_matrix.csv  ({_read_df.shape[0]} NBs x {_read_df.shape[1]} files)")

from matplotlib.colors import ListedColormap
cmap = ListedColormap([GREY, NAVY])
fig, ax = plt.subplots(figsize=(0.65 * len(files_axis) + 3, 0.34 * len(downstream_nbs) + 1.5))
ax.imshow(M, aspect='auto', cmap=cmap, vmin=0, vmax=1)
ax.set_xticks(range(len(files_axis)))
ax.set_xticklabels([f.split('/')[-1] for f in files_axis], rotation=55, ha='right', color=NAVY, fontsize=8)
ax.set_yticks(range(len(downstream_nbs)))
ax.set_yticklabels([p.stem for p in downstream_nbs], color=NAVY, fontsize=8)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        ax.text(j, i, 'Y' if M[i, j] == 1 else '.',
                ha='center', va='center', fontsize=7,
                color=CREAM if M[i, j] == 1 else NAVY)
ax.set_title('Which downstream NB reads which file  (NAVY=reads, GREY=does not)',
             color=NAVY, fontweight='bold', pad=10)
ax.tick_params(axis='both', which='both', length=0)
for sp in ax.spines.values():
    sp.set_edgecolor(NAVY)
plt.tight_layout()

<a id="sec-external"></a>
## 6. External data (`data/external/`)

- **`data/external/gbsa-study/`** — the upstream GBSA study package (RAW + DERIVED + a snapshot notebooks tree). Downstream code imports from here via `discovery9.paths.GBSA_STUDY`. **READ-ONLY** from this repo.
- **`data/external/gbsa-study-original/`** — the pristine upstream snapshot (same layout as `gbsa-study/`, kept for provenance). **READ-ONLY**; do not modify.

See section 2b for the raw CSVs. The derived tree is scanned below.


In [ ]:
print('### 6a. `data/external/gbsa-study/data/derived/`  (upstream derived; produced by upstream NBs)\n')

upstream_derived_top = sorted([p for p in GBSA_STUDY.joinpath('data/derived').glob('*.csv')])
widths = [72, 10, 10, 34]
hdr(['file (relative)', 'size KB', 'n_rows', 'first columns'], widths)
for p in upstream_derived_top:
    print_row([rel(p), f'{size_kb(p):.1f}', n_rows_csv(p) or '', ', '.join(cols_csv(p, n=4))], widths)

for sub in ('doe', 'study2', 'temporal'):
    print(f'\n--- data/external/gbsa-study/data/derived/{sub}/ ---')
    for p in sorted(GBSA_STUDY.joinpath('data/derived', sub).glob('*.csv')):
        print_row([rel(p), f'{size_kb(p):.1f}', n_rows_csv(p) or '', ', '.join(cols_csv(p, n=4))], widths)

print('\n--- producer map (upstream NBs and reproduce/upstream scripts) ---')
producer_map = [
    ('data/external/gbsa-study/data/derived/coverage.csv',                 'notebooks/04_coverage.ipynb'),
    ('data/external/gbsa-study/data/derived/coverage_by_bin.csv',          'notebooks/04_coverage.ipynb'),
    ('data/external/gbsa-study/data/derived/gbsa_vs_docking_per_target.csv','notebooks/06_gbsa_vs_docking.ipynb'),
    ('data/external/gbsa-study/data/derived/per_target_own_null.csv',      'notebooks/06_gbsa_vs_docking.ipynb'),
    ('data/external/gbsa-study/data/derived/bedroc_alpha_robustness.csv',  'notebooks/06_gbsa_vs_docking.ipynb'),
    ('data/external/gbsa-study/data/derived/panel_wilcoxon.csv',           'notebooks/06_gbsa_vs_docking.ipynb'),
    ('data/external/gbsa-study/data/derived/selection_analysis.csv',       'notebooks/13_selection_correction.ipynb'),
    ('data/external/gbsa-study/data/derived/sensitivity_4L7G.csv',         'notebooks/13_selection_correction.ipynb'),
    ('data/external/gbsa-study/data/derived/validation_power.csv',         'notebooks/13_selection_correction.ipynb'),
    ('data/external/gbsa-study/data/derived/validation_targets_locked.csv','notebooks/13_selection_correction.ipynb'),
    ('data/external/gbsa-study/data/derived/validation_lock.json',         'notebooks/13_selection_correction.ipynb'),
    ('data/external/gbsa-study/data/derived/locked_settings.csv',          'notebooks/13_selection_correction.ipynb'),
    ('data/external/gbsa-study/data/derived/physics_factor_importance.csv','notebooks/07_physics_importance.ipynb'),
    ('data/external/gbsa-study/data/derived/md_speed_honest.csv',          'notebooks/11_timestep_throughput.ipynb'),
    ('data/external/gbsa-study/data/derived/md_speed_importance.csv',      'notebooks/11_timestep_throughput.ipynb'),
    ('data/external/gbsa-study/data/derived/md_stability_by_dt.csv',       'notebooks/10_timestep_stability.ipynb'),
    ('data/external/gbsa-study/data/derived/md_usable_throughput.csv',     'notebooks/10_timestep_stability.ipynb'),
    ('data/external/gbsa-study/data/derived/md_usable_throughput_honest.csv','notebooks/10_timestep_stability.ipynb'),
    ('data/external/gbsa-study/data/derived/study2/*',                     'notebooks/09_study2_gromacs_screening.ipynb'),
    ('data/external/gbsa-study/data/derived/doe/*',                        'notebooks/08_doe_analysis.ipynb'),
    ('data/external/gbsa-study/data/derived/temporal/temporal_bedroc_vs_frames.csv', 'notebooks/12_temporal_convergence.ipynb + reproduce/upstream/temporal_convergence.py'),
    ('data/external/gbsa-study/data/derived/temporal/temporal_bedroc_vs_time.csv',   'notebooks/12_temporal_convergence.ipynb + reproduce/upstream/temporal_convergence.py'),
    ('data/external/gbsa-study/data/derived/temporal/per_target_settling.csv',       'notebooks/12_temporal_convergence.ipynb'),
    ('data/external/gbsa-study/data/derived/temporal/intdiel_per_target.csv',        'notebooks/12_temporal_convergence.ipynb'),
    ('data/external/gbsa-study/data/derived/temporal/intdiel_by_family.csv',         'notebooks/12_temporal_convergence.ipynb'),
    ('data/external/gbsa-study/data/derived/temporal/bedroc_all_combos_per_target.csv','reproduce/upstream/bedroc_doe.py (consumed by NB 14, 15, downstream reproduce/gbsa_param_vs_md*.py)'),
    ('data/external/gbsa-study/data/derived/temporal/bedroc_all_combos_summary.csv', 'reproduce/upstream/bedroc_doe.py'),
]
for f, prod in producer_map:
    print(f'  {f}')
    print(f'      producer: {prod}')

print('\n### 6b. `data/external/gbsa-study/data/external/`  (upstream external assets)')
up_ext = GBSA_STUDY / 'data' / 'external'
if up_ext.exists():
    for p in sorted(up_ext.iterdir()):
        print(f'    {rel(p)}   ({size_kb(p):.1f} KB)')

print('\n### 6c. `data/external/gbsa-study-original/`  (pristine upstream snapshot -- do not modify)')
print('    layout mirrors data/external/gbsa-study/; kept for provenance only.')

### 6d. `data/external/bayesopt/` — BayesOpt (Optuna) provenance

Vendored Bayesian-optimisation artifacts that picked the 270-set MD production `.mdp`.
Read by `notebooks/02_md_config_provenance.ipynb`. **READ-ONLY** from this repo.

- `bayes_opt.db` — SQLite Optuna study DB (~928 KB). Studies: `smoke_5hu9_ligand01_v1` (1 trial), `md_prod_v1` (200 COMPLETE), `md_prod_v2` (33 COMPLETE + 142 FAIL). All MAXIMIZE fitness = ns/day × stability. Winner: `md_prod_v1` trial 96, fitness 20.073.
- `bo_winner_v1.json` — the top-1 `.mdp` + FF triple, plus a `provenance` block (study name, trial number, fitness, extraction UTC, source complex 5HU9/ligand01).
- `code/` — the 17-parameter search-space source (`space_v2.py`, `space.py` v1, `objective.py`, `optimize.py`, `optimize_v2.py`, `verify_top.py`).
- `report/BO_report.ipynb`, `report/BO_report.html`, `report/reviewer_notes/` — reviewed BayesOpt report (10 iter × 5 reviewer rounds).


In [ ]:
print("### 6d. `data/external/bayesopt/`  (BayesOpt provenance for the 270-set MD .mdp)\n")

bo_root = EXTERNAL / 'bayesopt'
if bo_root.exists():
    widths = [64, 12, 34]
    hdr(['file / dir (relative)', 'size KB', 'role'], widths)
    roles = {
        'bayes_opt.db':        'Optuna SQLite study DB (3 studies)',
        'bo_winner_v1.json':   'extracted top-1 config + provenance',
        'code':                '17-param search-space + drivers',
        'report':              'reviewed BO report + reviewer_notes/',
    }
    for p in sorted(bo_root.iterdir()):
        kb = size_kb(p) if p.is_file() else sum(size_kb(x) for x in p.rglob('*') if x.is_file())
        print_row([rel(p), f'{kb:.1f}', roles.get(p.name, '')], widths)
    print()
    print('  producer: pipeline-mps/src/pipeline_mps/bayes_opt/  (Optuna TPE, MAXIMIZE fitness)')
    print('  consumer: notebooks/02_md_config_provenance.ipynb')
else:
    print('  (not present — data/external/bayesopt/ was not vendored in this checkout)')


<a id="sec-nbindex"></a>
## 7. Notebook index (reverse map)

For each notebook: what it CONSUMES (reads) and what it PRODUCES (writes). Given a NB name, this tells you what it touches on disk.

All references are grepped directly from the `.ipynb` source. `load_*` helpers are expanded to the underlying files.


In [ ]:
print('### 7a. Act 1 setup + entry index (notebooks/00..06)\n')

nb_index = [
    dict(nb='notebooks/00_data_catalog.ipynb',
         consumes='walks data/raw/, data/derived/, data/external/, tables/, figures/ (read-only)',
         produces='figures/00_data_catalog_fig1.png (heatmap)'),
    dict(nb='notebooks/01_scope_and_map.ipynb',
         consumes='data/external/gbsa-study/data/raw/newbench_targets.csv',
         produces='figures/01_scope_and_map_fig{1..2}.png'),
    dict(nb='notebooks/02_md_config_provenance.ipynb',
         consumes='data/external/bayesopt/{bayes_opt.db, bo_winner_v1.json, code/space_v2.py}',
         produces='figures/02_md_config_provenance_fig{1..2}.png'),
    dict(nb='notebooks/03_dataset_overview.ipynb',
         consumes='data/derived/features.parquet (via load_features)',
         produces='figures/03_dataset_overview_fig1.png'),
    dict(nb='notebooks/04_coverage.ipynb',
         consumes='data/external/gbsa-study/data/raw/{gbsa_dG_raw.csv, metadata.csv, md_productions_raw.csv}',
         produces='data/external/gbsa-study/data/derived/{coverage.csv, coverage_by_bin.csv}'),
    dict(nb='notebooks/05_docking_baseline.ipynb',
         consumes='data/external/gbsa-study/data/raw/metadata.csv',
         produces='figures/05_docking_baseline_fig1.png'),
    dict(nb='notebooks/06_gbsa_vs_docking.ipynb',
         consumes='data/external/gbsa-study/data/raw/{gbsa_dG_raw.csv, metadata.csv}',
         produces='data/external/gbsa-study/data/derived/{bedroc_alpha_robustness.csv, per_target_own_null.csv, gbsa_vs_docking_per_target.csv, panel_wilcoxon.csv}; figures/06_gbsa_vs_docking_fig1.png'),
]
widths = [50, 68, 60]
hdr(['NB', 'consumes', 'produces'], widths)
for e in nb_index:
    print_row([e['nb'], e['consumes'], e['produces']], widths)

print('\n### 7b. Act 2 data-side experiments (notebooks/07..13)\n')
nb_index_2 = [
    dict(nb='notebooks/07_physics_importance.ipynb',
         consumes='data/external/gbsa-study/data/raw/{gbsa_dG_raw.csv, metadata.csv}',
         produces='data/external/gbsa-study/data/derived/physics_factor_importance.csv; figures/07_physics_importance_fig{1..2}.png'),
    dict(nb='notebooks/08_doe_analysis.ipynb',
         consumes='data/external/gbsa-study/data/derived/{bedroc_all_combos_per_target.csv (temporal/), physics_factor_importance.csv, study2/wallclock_by_config.csv, etc.}',
         produces='data/external/gbsa-study/data/derived/doe/{study1_effects_{bedroc,tau}.csv, study2_{effects,anova}_{bedroc,wallclock}.csv} (6 files); figures/08_doe_analysis_s{1,2}_*.{png,pdf} (flat, 8 figure pairs)'),
    dict(nb='notebooks/09_study2_gromacs_screening.ipynb',
         consumes='data/external/gbsa-study/data/raw/{md_variants_manifest_raw.csv, md_variants_prod_perf_raw.csv, md_variants_gbsa_scores_raw.csv, metadata.csv}',
         produces='data/external/gbsa-study/data/derived/study2/*.csv (14 files); figures/09_study2_*.{png,pdf} (9 flat pairs)'),
    dict(nb='notebooks/10_timestep_stability.ipynb',
         consumes='data/external/gbsa-study/data/raw/{md_productions_raw.csv, md_speed_smoke_L40S.csv}',
         produces='data/external/gbsa-study/data/derived/{md_stability_by_dt.csv, md_usable_throughput.csv, md_usable_throughput_honest.csv}; figures/10_timestep_stability_fig1.png'),
    dict(nb='notebooks/11_timestep_throughput.ipynb',
         consumes='data/external/gbsa-study/data/raw/md_speed_smoke_L40S.csv',
         produces='data/external/gbsa-study/data/derived/{md_speed_honest.csv, md_speed_importance.csv}; figures/11_timestep_throughput_fig1.png'),
    dict(nb='notebooks/12_temporal_convergence.ipynb',
         consumes='data/external/gbsa-study/data/derived/{study2/reproducibility_floor_bedroc.csv, temporal/{bedroc_all_combos_per_target.csv, temporal_bedroc_vs_frames.csv, temporal_bedroc_vs_time.csv}}',
         produces='data/external/gbsa-study/data/derived/temporal/{intdiel_per_target.csv, per_target_settling.csv}; figures/12_temporal_convergence_trajectory_length.{png,pdf}'),
    dict(nb='notebooks/13_selection_correction.ipynb',
         consumes='data/external/gbsa-study/data/raw/{gbsa_dG_raw.csv, metadata.csv}',
         produces='data/external/gbsa-study/data/derived/{selection_analysis.csv, sensitivity_4L7G.csv, validation_power.csv, validation_targets_locked.csv, validation_lock.json, locked_settings.csv}; figures/13_selection_correction_fig1.png'),
]
hdr(['NB', 'consumes', 'produces'], widths)
for e in nb_index_2:
    print_row([e['nb'], e['consumes'], e['produces']], widths)

print('\n### 7c. Act 3 MD features on the discovery-9 (notebooks/14..21)\n')
nb_index_3 = [
    dict(nb='notebooks/14_per_complex_deep_dive.ipynb',
         consumes='data/derived/features.parquet; data/raw/complex_analyses/*/timeseries.parquet,ligand_rmsf.parquet,protein_ca_rmsf.parquet,contacts_persistence.tsv',
         produces='figures/14_per_complex_deep_dive_fig{1..5}.png'),
    dict(nb='notebooks/15_per_target_aggregation.ipynb',
         consumes='data/derived/features.parquet',
         produces='figures/15_per_target_aggregation_fig1.png'),
    dict(nb='notebooks/16_feature_distributions.ipynb',
         consumes='data/derived/features.parquet',
         produces='figures/16_feature_distributions_fig1.png'),
    dict(nb='notebooks/17_kinematic_phase_space.ipynb',
         consumes='data/derived/features.parquet',
         produces='figures/17_kinematic_phase_space_fig1.png'),
    dict(nb='notebooks/18_time_series_overlay.ipynb',
         consumes='data/derived/features.parquet; data/raw/complex_analyses/*/timeseries.parquet (via load_per_complex_analysis)',
         produces='figures/18_time_series_overlay_fig1.png'),
    dict(nb='notebooks/19_active_site_fingerprint.ipynb',
         consumes='data/derived/features.parquet; data/raw/complex_analyses/*/contacts_persistence.tsv',
         produces='figures/19_active_site_fingerprint_fig1.png'),
    dict(nb='notebooks/20_feature_correlation.ipynb',
         consumes='data/derived/features.parquet',
         produces='figures/20_feature_correlation_fig1.png'),
    dict(nb='notebooks/21_actives_decoys_separation.ipynb',
         consumes='data/derived/features.parquet',
         produces='figures/21_actives_decoys_separation_fig1.png'),
]
hdr(['NB', 'consumes', 'produces'], widths)
for e in nb_index_3:
    print_row([e['nb'], e['consumes'], e['produces']], widths)

print('\n### 7d. Act 4 ranking + verdict (notebooks/22..30)\n')
nb_index_4 = [
    dict(nb='notebooks/22_bedroc_baselines.ipynb',
         consumes='data/derived/features.parquet; data/derived/canonical_baselines.csv; data/external/gbsa-study/data/raw/{metadata.csv, gbsa_dG_raw.csv}',
         produces='figures/22_bedroc_baselines_fig1.png'),
    dict(nb='notebooks/23_ml_combo_selection.ipynb',
         consumes='data/derived/features.parquet; data/derived/deep_research_wide.csv; data/derived/canonical_baselines.csv',
         produces='figures/23_ml_combo_selection_fig{1..4}.png'),
    dict(nb='notebooks/24_full_factorial_ml.ipynb',
         consumes='data/derived/features.parquet; data/derived/canonical_baselines.csv; data/derived/hardened_claim_b.csv; data/derived/deep_research_wide.csv; data/external/gbsa-study/data/raw/{metadata.csv, gbsa_dG_raw.csv}; data/external/gbsa-study/data/derived/study2/bedroc20_partial.csv',
         produces='figures/24_full_factorial_ml_fig1.png'),
    dict(nb='notebooks/25_deep_research_verdict.ipynb',
         consumes='data/derived/features.parquet; data/derived/canonical_baselines.csv; data/external/gbsa-study/data/derived/temporal/{per_target_settling.csv, temporal_bedroc_vs_frames.csv, temporal_bedroc_vs_time.csv}',
         produces='figures/25_deep_research_verdict_fig1.png'),
    dict(nb='notebooks/26_gbsa_param_vs_md.ipynb',
         consumes='data/derived/features.parquet; data/derived/gbsa_param_vs_md*.csv; data/external/gbsa-study/data/derived/temporal/bedroc_all_combos_per_target.csv',
         produces='figures/26_gbsa_param_vs_md_fig{1..2}.png'),
    dict(nb='notebooks/27_ligand_chem_gbsa_surrogate.ipynb',
         consumes='data/derived/features.parquet; data/derived/ligand_chem.parquet; data/external/gbsa-study/data/raw/gbsa_dG_raw.csv; data/external/gbsa-study/data/derived/temporal/bedroc_all_combos_per_target.csv',
         produces='figures/27_ligand_chem_gbsa_surrogate_fig{1..4}.png'),
    dict(nb='notebooks/28_single_feature_bedroc.ipynb',
         consumes='data/derived/features.parquet; data/derived/hardened_claim_b.csv; data/derived/canonical_baselines.csv; data/derived/single_feature_bedroc.csv; data/external/gbsa-study/data/raw/{metadata.csv, gbsa_dG_raw.csv} (via load_metadata + load_gbsa)',
         produces='figures/28_single_feature_bedroc_fig{1..3}.png'),
    dict(nb='notebooks/29_rank_fusion_deployable.ipynb',
         consumes='data/derived/features.parquet; data/derived/ligand_chem.parquet; data/derived/canonical_baselines.csv; data/external/gbsa-study/data/raw/{metadata.csv, gbsa_dG_raw.csv}',
         produces='figures/29_rank_fusion_deployable_fig1.png'),
    dict(nb='notebooks/30_family_stratification.ipynb',
         consumes='data/derived/features.parquet; data/derived/deep_research_wide.csv; data/derived/single_feature_bedroc.csv; data/derived/canonical_baselines.csv; data/external/gbsa-study/data/raw/metadata.csv (via load_metadata)',
         produces='figures/30_family_stratification_fig{1..2}.png'),
]
hdr(['NB', 'consumes', 'produces'], widths)
for e in nb_index_4:
    print_row([e['nb'], e['consumes'], e['produces']], widths)


<a id="sec-cheat"></a>
## 8. Quick-reference cheat sheet

Common lookups when you can't remember where a thing lives:


In [ ]:
cheat = [
    ('Where is the canonical GBSA-locked baseline value?',
     'data/derived/canonical_baselines.csv  --  row baseline_name = gbsa_locked_9T_imputed  (panel_bedroc = 0.5412)'),
    ('Where are the 4A5S active/decoy labels?',
     'data/external/gbsa-study/data/raw/metadata.csv  --  MANIFEST.tsv has them empty; use metadata.csv as source of truth.'),
    ('Which NB produces the DOE figures?',
     'notebooks/08_doe_analysis.ipynb  -->  figures/08_doe_analysis_s{1,2}_*.{png,pdf} (flat, no subdir)'),
    ('Which script regenerates the canonical baselines?',
     'reproduce/canonical_baselines.py  (also `--verify` mode to re-check without overwriting)'),
    ('Which script regenerates panel_bedroc_summary.csv + rank_fusion_sweep.csv?',
     'reproduce/generate_summary_tables.py'),
    ('Which NB holds the P1/P2/P3 verdict?',
     'notebooks/25_deep_research_verdict.ipynb'),
    ('Which file has the per-(GBSA-combo, target) BEDROC (all 48 combos)?',
     'data/external/gbsa-study/data/derived/temporal/bedroc_all_combos_per_target.csv  (produced by reproduce/upstream/bedroc_doe.py; consumed by NB 26, 27 and downstream reproduce/gbsa_param_vs_md*.py)'),
    ('Which file has the per-complex GBSA dG at every combo?',
     'data/external/gbsa-study/data/raw/gbsa_dG_raw.csv  (rename mean_dG_kcalmol -> gbsa_dG; use discovery9.io.load_gbsa or load_gbsa_all)'),
    ('Where is the master feature table used by every downstream ML NB?',
     'data/derived/features.parquet  (270 x ~68 cols; load via discovery9.io.load_features which auto-merges ligand_chem.parquet)'),
    ('Where do per-complex analysis outputs live (timeseries, RMSF, contacts)?',
     'data/raw/complex_analyses/<target>/<complex_id>/{summary.json, timeseries.parquet, hbond_timeseries.parquet, ifp_timeseries.parquet, ligand_rmsf.parquet, protein_ca_rmsf.parquet, contacts_persistence.tsv, hbonds_persistence.tsv}'),
    ('Which NB reads the SI per-target PDFs?',
     'None -- figures/SI/pdf/*.pdf are output artifacts. Builder: reproduce/bundle_SI_pdf.py + reproduce/make_SI_per_target.py'),
    ('Where is the study-2 BEDROC control matrix (bedroc20_partial)?',
     'data/external/gbsa-study/data/derived/study2/bedroc20_partial.csv  (consumer: NB 24 + discovery9.io.load_bedroc_matrix)'),
    ('Where is the NewBench-27 scope table?',
     'data/external/gbsa-study/data/raw/newbench_targets.csv  (9 discovery + 18 validation; only consumer: NB 01)'),
    ('Where does hardened_claim_b live and what verdict does it hold?',
     'data/derived/hardened_claim_b.csv + hardened_claim_b_summary.md  (producer: reproduce/hardened_claim_b.py; consumer: NB 28, 30)'),
    ('Where is the review log for iter 5?',
     'reviews/iter_05/{BASELINE_RECONCILIATION_LOG.md, BRIEF_FOR_FRESH_REVIEWERS.md, DOC_UPDATE_LOG.md, R1..R5_*.md, SYNTHESIS.md, FLATTEN_MAPPING.md, FLATTEN_LOG.md}'),
]

for q, a in cheat:
    print(f'Q. {q}')
    print(f'   -> {a}\n')

<a id="sec-drift"></a>
## 9. Known inconsistencies (schema drift, doc drift)

Flagged, not fixed. Each entry was found by cross-referencing the notebooks / `reproduce` scripts against what actually lives on disk.


In [ ]:
drift = [
    ('doc drift',
     'notebooks/01_scope_and_map.ipynb (Section B, cross-reference block) refers to `tables/canonical_baselines.csv`, '
     'but the file actually lives at `data/derived/canonical_baselines.csv`. Do not fix in this NB; flagged for a doc pass.'),
    ('CSV header convention',
     '`data/derived/canonical_baselines.csv`, `tables/panel_bedroc_summary.csv`, `tables/rank_fusion_sweep.csv` all use `#` comment lines at the top for the regeneration command. Load with `pd.read_csv(..., comment="#")`. Not every consumer NB does this uniformly.'),
    ('column set drift',
     '`data/derived/features.parquet` has 68 columns; `notebooks/23_ml_combo_selection.ipynb` (execution log) reports 81 columns when loaded via load_features(with_ligand_chem=True) '
     '(features.parquet 68 + ligand_chem.parquet 15 - shared keys 2 = 81). This is expected -- documenting the arithmetic here so future readers do not chase a phantom bug.'),
    ('panel_bedroc_summary duality (iter 5 -> iter 6 reconcile)',
     '`tables/panel_bedroc_summary.csv` currently contains BOTH canonical (9T-imputed) rows AND `_8T_legacy` rows from the pre-reconcile runs. Rows are labelled; both coexist by design. See reviews/iter_05/BASELINE_RECONCILIATION_LOG.md.'),
    ('MANIFEST.tsv vs metadata.csv',
     'When upstream MANIFEST.tsv is empty for 4A5S, all consumers use `data/external/gbsa-study/data/raw/metadata.csv` as the source of truth for is_active. Do NOT trust MANIFEST.tsv alone.'),
    ('producer script unknown for two derived files',
     '`data/derived/md_surrogate_bedroc.csv` and `data/derived/single_feature_bedroc.csv` do not have an obvious producer script in `reproduce/`. '
     'Marked "producer: UNKNOWN -- investigate" in section 3. `single_feature_bedroc.csv` is CONSUMED by NB 28 + NB 30 + NB 01, so it is not an orphan; `md_surrogate_bedroc.csv` appears to be an orphan.'),
    ('two versions of gbsa_param_vs_md',
     'Both `reproduce/gbsa_param_vs_md.py` (v1) and `reproduce/gbsa_param_vs_md_v2.py` (v2) exist and both write to `data/derived/`. NB 26 consumes the v2 outputs; v1 output `gbsa_param_vs_md_correlations.csv` is retained for provenance but not read by any NB currently in the tree.'),
]

for kind, msg in drift:
    print(f'[{kind}]')
    print(f'  {msg}\n')

<a id="sec-orphans"></a>
## 10. Orphans + unresolved items

Files on disk with no active consumer, plus files that lack a documented producer.


In [ ]:
print('### Orphans (produced -> not consumed)\n')
orphans = [
    ('data/derived/md_surrogate_bedroc.csv',
     'no producer script found; no NB currently reads it. Suspect it is an earlier NB-27 hand-run artifact.'),
    ('data/derived/gbsa_param_vs_md_correlations.csv',
     'produced by reproduce/gbsa_param_vs_md.py (v1); NB 26 now consumes only the v2 outputs. Kept for provenance.'),
    ('data/external/gbsa-study/data/raw/md_variants_gbsa_scores_raw.csv.pre-rerun-backup',
     'explicit backup file; no consumer.'),
    ('data/derived/pbc_qc.csv',
     'consumed only by review docs (reviewer QC pass); not by any NB.'),
]
for f, note in orphans:
    print(f'  {f}')
    print(f'      note: {note}\n')

print('### Producer UNKNOWN\n')
unknown = [
    ('data/derived/md_surrogate_bedroc.csv',
     'no matching to_csv() call in reproduce/ or notebooks/. Likely an earlier hand-run.'),
    ('data/derived/single_feature_bedroc.csv',
     'no matching to_csv() call in reproduce/ or notebooks/. Likely written by NB 28 in an earlier iteration and then version-controlled.'),
]
for f, note in unknown:
    print(f'  {f}')
    print(f'      note: {note}\n')

<a id="sec-offrepo"></a>
## 11. Off-repo storage

Big binary artifacts that live outside the repo (STORE trees, workspaces). Not version-controlled here; listed so you know where to find them.

- **STORE trajectory tree** — `/mnt/netapp1/Store_othcxlwa/lwa_transfer/00_PRIORITY_discovery9_trajectories/`.
  9 targets × 30 ligands × 1 production = 270 GROMACS 30-ns trajectories, ~237 GB. This is the raw source that `reproduce/analyze_complex.py` reads to regenerate `data/raw/complex_analyses/`.
- **Tier-3 winner-replica workspace** — `/mnt/netapp1/Store_othcxlwa/pipeline-mps-workspaces/discovery9_winner/`.
  Planned confirmatory tree: 8 targets × 30 ligands × 3 replicas × 20 ns from the BayesOpt winner `.mdp`. Not yet in this repo; queued for when compute is available.
- **BayesOpt trial workspaces** — `/mnt/netapp1/Store_othcxlwa/pipeline-mps-workspaces/bayes_opt/{md_prod_v1,md_prod_v2,seeds}/`.
  Per-trial workspaces from the Optuna runs. Only the SQLite DB (`bayes_opt.db`) and the winner JSON are vendored under `data/external/bayesopt/`; the per-trial `.mdp`, topologies and `.log` outputs live here.
- **Safety copy of BO artifacts** — `/mnt/netapp1/Store_othcxlwa/lwa_transfer/projects/07_pipeline-mps-bayesopt/`.
  Safety copy of `bayes_opt.db` + BO notebooks + configs. Same content as `data/external/bayesopt/` at the time of vendoring.


In [ ]:
print('### 11. Off-repo storage — big binary artifacts not vendored in this repo\n')

offrepo = [
    ('/mnt/netapp1/Store_othcxlwa/lwa_transfer/00_PRIORITY_discovery9_trajectories/',
     'STORE trajectory tree: 270 productions (9 targets x 30 ligands), ~237 GB'),
    ('/mnt/netapp1/Store_othcxlwa/pipeline-mps-workspaces/discovery9_winner/',
     'Tier-3 winner-replica workspace: planned 8 x 30 x 3, NOT yet computed'),
    ('/mnt/netapp1/Store_othcxlwa/pipeline-mps-workspaces/bayes_opt/md_prod_v1/',
     'BayesOpt trial workspace v1 (200 trials)'),
    ('/mnt/netapp1/Store_othcxlwa/pipeline-mps-workspaces/bayes_opt/md_prod_v2/',
     'BayesOpt trial workspace v2 (34 trials)'),
    ('/mnt/netapp1/Store_othcxlwa/pipeline-mps-workspaces/bayes_opt/seeds/',
     'BayesOpt seed topologies'),
    ('/mnt/netapp1/Store_othcxlwa/lwa_transfer/projects/07_pipeline-mps-bayesopt/',
     'Safety copy of BO DB + notebooks + configs'),
]
for path, desc in offrepo:
    print(f'  {path}')
    print(f'      note: {desc}')

print('\nAll paths are on the netapp1 STORE tree; this repo intentionally does not vendor')
print('trajectory data or per-trial workspaces. Consult with the compute allocation owner')
print('before touching or copying these paths.')

In [ ]:
# --- export figures (same pattern as NBs 11-19) ---
try:
    FIGURES.mkdir(parents=True, exist_ok=True)
except NameError:
    from discovery9.paths import FIGURES
    FIGURES.mkdir(parents=True, exist_ok=True)
try:
    _cream = CREAM
except NameError:
    from discovery9.style import CREAM as _cream
figs = list(globals().get('_SAVED_FIGS', []))
for num in plt.get_fignums():
    f = plt.figure(num)
    if f not in figs:
        figs.append(f)
saved = []
for i, fig in enumerate(figs, start=1):
    out = FIGURES / f"{NB_STEM}_fig{i}.png"
    try:
        fig.savefig(out, bbox_inches='tight', dpi=300, facecolor=_cream)
    except Exception as e:
        print(f'  WARN: failed to save fig{i}: {e}')
        continue
    saved.append(str(out.name))
print(f'saved {len(saved)} figures:')
for s in saved:
    print(' ', s)